# End-to-End: Rotated Covariance (Haar Orthogonal)

- This config builds `Sigma = U diag(lambda) U^T` with Haar-distributed `U`.
- Rotation changes eigenvectors but not eigenvalues, so spectral recovery targets the same eigenvalue law.

**Config:** `configs/simulation_examples/03_constant_rotated_haar.yaml`


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'mpdiff').exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate project root (expected pyproject.toml and src/mpdiff).')
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from mpdiff.config.loader import load_config
from mpdiff.experiments.run_end_to_end import run_end_to_end

config_path = PROJECT_ROOT / 'configs/simulation_examples/03_constant_rotated_haar.yaml'
cfg = load_config(config_path)
out_dir = Path(cfg.global_settings.output_dir)
out_dir


In [ ]:
summary = run_end_to_end(config_path)
summary


In [ ]:
summary_csv = out_dir / 'full_pipeline_method_summary.csv'
metadata_json = out_dir / 'full_pipeline_metadata.json'

df = pd.read_csv(summary_csv)
display(df)

metadata = json.loads(metadata_json.read_text())
print('timers (seconds):')
for k, v in metadata['timers_seconds'].items():
    print(f'  {k}: {v:.4f}')


In [ ]:
fig_paths = [
    out_dir / 'full_pipeline_overlay_population_empirical_recovered.png',
    out_dir / 'full_pipeline_overlay_observed_reconstructed_forward.png',
    out_dir / 'full_pipeline_eigen_histograms.png',
    out_dir / 'full_pipeline_method_population_comparison.png',
]

for p in fig_paths:
    if p.exists():
        img = mpimg.imread(p)
        plt.figure(figsize=(10, 4.5))
        plt.imshow(img)
        plt.title(p.name)
        plt.axis('off')
        plt.show()


## Interpretation

- Expected behavior: recovered spectrum aligns with the underlying diagonal-eigenvalue law, independent of `U`.
- Discrepancies mainly come from finite `d,n` and inversion regularization, not from rotation itself.

Cross-check both population-recovery metrics and observed-reconstruction metrics before concluding model quality.
